<style>
  /* 1. Fix standard input code cells */
  pre, code, .jp-CodeCell .jp-Cell-inputWrapper {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
  }

  /* 2. Fix display() outputs and rendered Markdown */
  .jp-OutputArea-output, 
  .jp-OutputArea-output pre, 
  .jp-RenderedHTMLCommon pre, 
  .jp-RenderedMarkdown pre {
      white-space: pre-wrap !important;
      word-wrap: break-word !important;
      overflow-wrap: break-word !important;
      
      /* Forces the browser to print the overflow instead of hiding it in a scrollbox */
      overflow-x: visible !important; 
      max-width: 100% !important;
  }
</style>

In [ ]:
%load_ext autoreload
%autoreload 2

### **Prediksi *Pavement Condition Index* menggunakan Model *Machine Learning (RF, XGBoost, ANN)***

#### 1. Pembersihan data dan rekayasa fitur

In [ ]:
# Memilih dataset dan parameter pemisahan
from IPython.display import display, Markdown
from pci_ml.data import list_datasets, load_data
from pci_ml.memory import configure
from pci_ml.ui import build_panel, format_dataframe_table

EXPORT_MODE = True   # True = print panel values instead of showing widgets
RELEASE_MEMORY = True

DEFAULTS = {
    "file": None,        # None = first dataset, or e.g. "data_2026.xlsx"
    "n_seed": 200,
    "test_size": 0.2,
    "share_tol": 0.01,
    "pci_min": 0,
}

configure(release=RELEASE_MEMORY)

datasets = list_datasets()
if DEFAULTS["file"] is not None and DEFAULTS["file"] not in datasets:
    raise ValueError(f"DEFAULTS['file'] '{DEFAULTS['file']}' not found in datasets")

df = None

def on_select(name):
    raw = None if name is None else load_data(name)
    handle.update(Markdown(format_dataframe_table(raw)))

panel = build_panel(datasets, DEFAULTS, on_file_change=on_select, export=EXPORT_MODE)

handle = display(Markdown(""), display_id=True)

on_select(panel.values()["file"])


In [ ]:
# Membersihkan data, menghitung fitur density, dan menambahkan fitur agregat
from IPython.display import display, Markdown
from pci_ml.data import add_aggregate_features, clean_data, load_data
from pci_ml.ui import format_dataframe_table

cfg = panel.values()
filename = cfg["file"]
df, features = clean_data(load_data(filename))
df, features = add_aggregate_features(df, features)

if not EXPORT_MODE:
    print(f"Parameter: file {filename} | jumlah seed {cfg['n_seed']} | "
          f"test size {cfg['test_size']} | toleransi {cfg['share_tol']} | "
          f"PCI minimum {cfg['pci_min']}")

print(f"Data telah dibersihkan; {len(features)} fitur (density + agregat) telah dihitung.")
display(Markdown(format_dataframe_table(df)))


#### 2. Pemisahan data latih dan data uji

In [ ]:
# Memisah data sesuai kriteria
from pci_ml.data import add_aggregate_features, clean_data, load_data, split_data
from pci_ml.memory import release
from pci_ml.ui import split_progress

if globals().get("df") is None:
    df, features = clean_data(load_data(filename))
    df, features = add_aggregate_features(df, features)

result = split_data(df, features, filename, {
    "n_seed": cfg["n_seed"],
    "test_size": cfg["test_size"],
    "share_tol": cfg["share_tol"],
    "pci_min": cfg["pci_min"],
}, progress=split_progress)
print()

# Print the summary outside the progress line so it always shows up
print(result["summary"])

release("result", "df")


#### 3. Pelatihan model *machine learning*

In [ ]:
# Mempersiapkan input model
import numpy as np
from pci_ml.data import (AGGREGATE_FEATURES, METADATA, PCI, ROAD_ID,
                         latest_checkpoint, load_checkpoint)
from pci_ml.memory import release

SEED = 42   # model seed (RF, XGBoost, Keras), separate from the split seed in meta["seed"]

# Memuat checkpoint terbaru untuk file yang dipilih
ckpt_tag = latest_checkpoint(filename)

train_df, test_df, features, meta = load_checkpoint(filename, ckpt_tag)
missing = [name for name in AGGREGATE_FEATURES if name not in features]
if missing:
    raise ValueError(
        "Checkpoint belum memuat fitur agregat. Jalankan ulang Cell 7. "
        f"Kolom hilang: {missing}"
    )

X_train = train_df[features].to_numpy(dtype="float32")
X_test = test_df[features].to_numpy(dtype="float32")
y_train = train_df[PCI].to_numpy(dtype="float32")    # PCI scale 0-100; divide by 100 in the ANN cell
y_test = test_df[PCI].to_numpy(dtype="float32")

groups_train = train_df[ROAD_ID].to_numpy()          # for GroupKFold when tuning
test_ids = test_df[[ROAD_ID, *METADATA]].reset_index(drop=True)

assert all(np.isfinite(a).all() for a in (X_train, X_test, y_train, y_test)), "NaN/inf found"

del train_df, test_df

n_tr, n_te = len(X_train), len(X_test)
total = n_tr + n_te
split_text = (f"**Train:** {X_train.shape} ({n_tr / total:.2%}) | "
              f"**Test:** {X_test.shape} ({n_te / total:.2%})")

# Staleness guard: the newest split may have been made with different parameters
current = {key: cfg[key] for key in ("n_seed", "test_size", "share_tol", "pci_min")}
if current != meta["params"]:
    print("PERINGATAN: split terbaru dibuat dengan parameter berbeda. "
          "Jalankan ulang Cell 7 untuk memisahkan data dengan parameter saat ini.")
    print(f"  Parameter saat ini : {current}")
    print(f"  Parameter split    : {meta['params']}")

display(Markdown(f"**Split terpilih:** `{ckpt_tag}`\n\n{split_text}"))

release("df", "ckpt_tag", "current")


In [ ]:
# Tuning hyperparameter dan pelatihan model Random Forest final
from IPython.display import HTML, Markdown, display
from pci_ml.experiments import evaluate_oof, evaluate_predictions
from pci_ml.random_forest import tune_random_forest
from pci_ml.ui import build_hyperparameter_table, build_param_panel, format_table

TUNING_DEFAULTS = {
    "n_iter": 100,
    "cv_splits": 5,
}

TUNING_SPECS = {
    "n_iter":    {"type": "int", "description": "Jumlah kandidat:", "label": "Jumlah kandidat"},
    "cv_splits": {"type": "int", "description": "Jumlah fold:", "label": "Jumlah fold"},
}

RF_SEARCH_SPACE = {
    "n_estimators": [300],
    "max_depth": [None, 20, 30],
    "min_samples_split": [2, 4, 6],
    "min_samples_leaf": [1, 2],
    "max_features": [0.1, 0.15, 0.2, 0.25, 0.3, 0.4],
    "max_samples": [0.5, 0.6, 0.7, 0.8, 0.9],
}

tuning_panel = build_param_panel(
    TUNING_DEFAULTS, TUNING_SPECS, "Parameter Tuning Random Forest:", export=EXPORT_MODE
)
tuning_params = tuning_panel.values()

display(Markdown(
    f"**Memulai tuning:** {tuning_params['n_iter']} kandidat | "
    f"{tuning_params['cv_splits']} fold CV"
))
rf_result = tune_random_forest(
    X_train, y_train, groups_train,
    RF_SEARCH_SPACE,
    n_iter=tuning_params["n_iter"],
    cv_splits=tuning_params["cv_splits"],
    random_state=SEED,
)

rf_best_params = rf_result["params"]
tuned_model = rf_result["model"]
rf_cv = evaluate_oof(
    y_train, rf_result["oof"], groups_train, rf_result["folds"]
)["pooled"]
tuned_test = evaluate_predictions(y_test, tuned_model.predict(X_test))

param_explanations = {
    "n_estimators": "Jumlah pohon keputusan.",
    "max_depth": "Kedalaman maksimum tiap pohon.",
    "min_samples_split": "Jumlah sampel minimum agar sebuah node boleh dipecah.",
    "min_samples_leaf": "Jumlah sampel minimum di tiap daun.",
    "max_features": "Fraksi fitur yang dipertimbangkan di tiap pohon.",
    "max_samples": "Fraksi data latih yang dipakai untuk tiap pohon.",
}

param_table = build_hyperparameter_table(
    rf_best_params, RF_SEARCH_SPACE.keys(), param_explanations
)

evaluation_table = format_table(
    ["Evaluasi", "MAE", "RMSE", "R²"],
    [
        ["CV pooled OOF", f"{rf_cv['mae']:.4f}", f"{rf_cv['rmse']:.4f}", f"{rf_cv['r2']:.4f}"],
        ["Test", f"{tuned_test['mae']:.4f}", f"{tuned_test['rmse']:.4f}", f"{tuned_test['r2']:.4f}"],
    ],
    aligns=["left", "right", "right", "right"],
)

n_test_roads = test_ids["ROAD_ID"].nunique()
explanation = f"""
**Keterangan**
- **MAE**: rata-rata selisih absolut antara prediksi dan PCI aktual, dalam satuan poin PCI. Semakin kecil semakin baik.
- **RMSE**: akar rata-rata kuadrat error, dalam satuan poin PCI. Lebih sensitif terhadap error besar; semakin kecil semakin baik.
- **R²**: proporsi variasi PCI yang dijelaskan model. Semakin mendekati 1 semakin baik.
- **CV pooled OOF**: semua prediksi out-of-fold dari {tuning_params['cv_splits']} fold digabung, lalu dihitung sekali. Ini estimasi generalisasi yang lebih stabil daripada rata-rata R² per fold, tetapi sedikit optimistis karena hyperparameter dipilih dari CV yang sama.
- **Test**: evaluasi pada data uji yang tidak dipakai saat tuning. Ini estimasi akhir pada {n_test_roads} road holdout, tetapi hanya satu holdout sehingga variansnya lebih tinggi.

Model final memakai hyperparameter terbaik dari tuning, dilatih pada seluruh data latih, lalu dievaluasi dengan GroupKFold (OOF) dan data uji.
"""

display(Markdown("### Hyperparameter terbaik"))
display(HTML(param_table))
display(Markdown("### Hasil evaluasi"))
display(HTML(evaluation_table))
display(Markdown(explanation))


### *Source code* modul dan fungsi

- `src/pci_ml/data.py` - pembersihan data, rekayasa fitur agregat, split, dan checkpoint.
- `src/pci_ml/random_forest.py` - factory estimator dan tuning Random Forest.
- `src/pci_ml/experiments.py` - fold, OOF, metrik, bobot, dan kalibrasi yang model-agnostic.
- `src/pci_ml/ui.py` - panel, progress split, dan tabel HTML.
- `src/pci_ml/memory.py` - manajemen memori notebook.